# M33 Notebook 1: Make the RGB mosaic

Read in the flux maps and combine them into an RGB mosaic of the galaxy, with zoom ins showing certain regions.

In [4]:
import os
import numpy as np

from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import AsinhStretch, PercentileInterval
from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd
from reproject import reproject_interp

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from astropy.coordinates import SkyCoord
import astropy.units as u

plt.rcParams["text.usetex"] = False

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=20)

In [5]:
def save_color_mosaic(
    directory_maps,
    color_to_lines,
    labels=None,
    color_rgb=None,
    cmap=None,
    attempt="v1",
    output_basename="M33-ColorComposite",
    require_all_lines_per_tile=True,
    # ---- zoom options ----
    zoom_center_pix=None,
    zoom_center_sky=None,
    zoom_size=250,
    zoom_loc="upper right",
    zoom_inset_frac=0.35,
    zoom_box_color="white",
    zoom_box_lw=1.5,
    zoom_mark=True,
    zooms=None,
    # ---- NEW: per-channel display controls ----
    channel_percentile=None,        # dict: {color: pct} e.g. {"blue":99.9, "green":99.5, "red":99.0}
    channel_gamma=None,             # dict: {color: gamma} e.g. {"blue":0.6, "green":1.0, "red":0.8}
    channel_sky_percentile=5.0,     # float or dict: sky subtraction percentile (default 5)
):
    """
    Build a WCS-aligned mosaic where each "color" can be the sum of multiple emission-line maps.
    Produces an RGB composite and an annotated PDF preview with optional zoom insets.

    NEW controls:
      - channel_percentile: per-color upper percentile for stretching (PercentileInterval)
      - channel_gamma: per-color gamma correction after stretching (I -> I**gamma)
      - channel_sky_percentile: per-color sky percentile (or scalar) for background subtraction
    """

    # --- normalize inputs: ensure each dict value is a list ---
    color_to_lines_norm = {}
    for color, lines in color_to_lines.items():
        if isinstance(lines, (list, tuple, set)):
            color_to_lines_norm[color] = list(lines)
        else:
            color_to_lines_norm[color] = [lines]

    # --- default labels ---
    if labels is None:
        labels = {c: " + ".join(color_to_lines_norm[c]) for c in color_to_lines_norm}

    # --- default color palette ---
    default_palette = {
        "red": (1.0, 0.0, 0.0),
        "green": (0.0, 1.0, 0.0),
        "blue": (0.0, 0.0, 1.0),
        "cyan": (0.0, 1.0, 1.0),
        "magenta": (1.0, 0.0, 1.0),
        "yellow": (1.0, 1.0, 0.0),
        "orange": (1.0, 0.55, 0.0),
        "purple": (0.6, 0.2, 0.8),
        "white": (1.0, 1.0, 1.0),
    }

    if color_rgb is None:
        color_rgb = {}

    # Resolve each color -> rgb triple
    resolved_rgb = {}
    for c in color_to_lines_norm:
        if c in color_rgb:
            resolved_rgb[c] = tuple(color_rgb[c])
        elif c.lower() in default_palette:
            resolved_rgb[c] = default_palette[c.lower()]
        else:
            try:
                from matplotlib.colors import to_rgb
                resolved_rgb[c] = to_rgb(c)
            except Exception:
                resolved_rgb[c] = (1.0, 1.0, 1.0)

    # --- Collect per-color inputs for mosaicking ---
    inputs_by_color = {c: [] for c in color_to_lines_norm}

    folders = [f for f in os.listdir(directory_maps) if f != ".DS_Store"]
    folders.sort()

    for folder in folders:
        folder_path = os.path.join(directory_maps, folder)
        folder_end = folder_path[-2:]  # your convention

        needed_paths = []
        per_color_paths = {}
        for c, lines in color_to_lines_norm.items():
            line_paths = [os.path.join(folder_path, f"M33{folder_end}-{ln}.fits") for ln in lines]
            per_color_paths[c] = line_paths
            needed_paths.extend(line_paths)

        if require_all_lines_per_tile:
            if not all(os.path.exists(p) for p in needed_paths):
                print(f"Skipping {folder_end} (missing one or more required files)")
                continue

        ref_path = next((p for p in needed_paths if os.path.exists(p)), None)
        if ref_path is None:
            print(f"Skipping {folder_end} (no files found)")
            continue

        href = fits.open(ref_path)[0]
        wref = WCS(href.header)

        for c, line_paths in per_color_paths.items():
            datas = []
            for p in line_paths:
                if not os.path.exists(p):
                    if require_all_lines_per_tile:
                        datas = []
                        break
                    else:
                        continue

                h = fits.open(p)[0]
                d = np.where(np.isfinite(h.data), h.data / 1e-17, np.nan)
                datas.append(d)

            if len(datas) == 0:
                continue

            summed = np.nansum(np.stack(datas, axis=0), axis=0)
            # use WCS from the first file that contributed to this summed image
            w_this = None
            for p in line_paths:
                if os.path.exists(p):
                    w_this = WCS(fits.getheader(p))
                    break

            if w_this is None:
                continue

            inputs_by_color[c].append((summed, w_this))


    valid_colors = [c for c in inputs_by_color if len(inputs_by_color[c]) > 0]
    if not valid_colors:
        raise ValueError("No valid inputs found for any color.")

    wcs_out, shape_out = find_optimal_celestial_wcs(inputs_by_color[valid_colors[0]])

    mosaics = {}
    for c in valid_colors:
        plane, _ = reproject_and_coadd(
            inputs_by_color[c],
            wcs_out,
            shape_out=shape_out,
            reproject_function=reproject_interp,
        )
        mosaics[c] = plane

    # ---- NEW: setup per-channel defaults ----
    if channel_percentile is None:
        channel_percentile = {}
    if channel_gamma is None:
        channel_gamma = {}

    def get_sky_pct(color):
        if isinstance(channel_sky_percentile, dict):
            return float(channel_sky_percentile.get(color, 5.0))
        return float(channel_sky_percentile)

    def get_hi_pct(color):
        return float(channel_percentile.get(color, 99.5))

    def get_gamma(color):
        return float(channel_gamma.get(color, 1.0))

    # --- normalization now takes per-color parameters ---
    def normalize(x, hi_pct=99.5, sky_pct=5.0, gamma=1.0):
        x = np.nan_to_num(x, nan=0.0)

        pos = x[x > 0]
        if pos.size > 0:
            sky = np.percentile(pos, sky_pct)
        else:
            sky = 0.0

        x = x - sky
        x[x < 0] = 0.0

        # stretch to [0,1] based on per-channel high percentile
        stretch = AsinhStretch() + PercentileInterval(hi_pct)
        x = stretch(x)
        x = np.clip(x, 0, 1)

        # gamma correction
        if gamma != 1.0:
            x = np.power(x, gamma)

        return np.clip(x, 0, 1)

    # --- additive color mixing into RGB ---
    rgb = np.zeros((shape_out[0], shape_out[1], 3), dtype=float)
    for c in valid_colors:
        In = normalize(
            mosaics[c],
            hi_pct=get_hi_pct(c),
            sky_pct=get_sky_pct(c),
            gamma=get_gamma(c),
        )
        cr, cg, cb = resolved_rgb[c]
        rgb[..., 0] += In * cr
        rgb[..., 1] += In * cg
        rgb[..., 2] += In * cb

    rgb = np.clip(rgb, 0, 1)

    # optional: mask background where all planes are zero-ish
    total = np.zeros(shape_out, dtype=float)
    for c in valid_colors:
        total += np.nan_to_num(mosaics[c], nan=0.0)
    rgb[np.isnan(total) | (total == 0)] = 0.0

    # --- Plot and save PDF preview ---
    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(projection=wcs_out)
    ax.coords["ra"].set_ticklabel(size=8)
    ax.coords["dec"].set_ticklabel(size=8)
    ax.imshow(rgb, origin="lower")
    ax.set_xlabel("RA", fontsize=10)
    ax.set_ylabel("Dec", fontsize=10)

    # -------------------------
    # Zoom insets (optional)
    # -------------------------
    if zooms is None:
        zooms = []

    # Backward-compat: if user still passes the old single-zoom args, convert to one-item list
    if ((zoom_center_pix is not None) or (zoom_center_sky is not None)) and (len(zooms) == 0):
        zooms = [dict(
            center_pix=zoom_center_pix,
            center_sky=zoom_center_sky,
            size=zoom_size,
            inset_pos=None,
            loc=zoom_loc,
            frac=zoom_inset_frac,
            color=zoom_box_color,
            lw=zoom_box_lw,
            mark=zoom_mark,
        )]

    for zspec in zooms:
        size = zspec.get("size", zoom_size)
        box_color = zspec.get("color", zoom_box_color)
        lw = zspec.get("lw", zoom_box_lw)
        mark = zspec.get("mark", zoom_mark)

        center_pix = zspec.get("center_pix", None)
        center_sky = zspec.get("center_sky", None)

        if (center_pix is None) and (center_sky is None):
            continue

        if center_pix is not None:
            xc, yc = center_pix
        else:
            if isinstance(center_sky, SkyCoord):
                sc = center_sky
            else:
                ra_deg, dec_deg = center_sky
                sc = SkyCoord(ra_deg * u.deg, dec_deg * u.deg, frame="icrs")
            xc, yc = wcs_out.world_to_pixel(sc)

        half = size / 2.0
        x0 = int(np.floor(xc - half))
        x1 = int(np.ceil(xc + half))
        y0 = int(np.floor(yc - half))
        y1 = int(np.ceil(yc + half))

        H, W = rgb.shape[0], rgb.shape[1]
        x0c, x1c = max(0, x0), min(W, x1)
        y0c, y1c = max(0, y0), min(H, y1)

        if (x1c - x0c) < 2 or (y1c - y0c) < 2:
            continue

        rect = plt.Rectangle(
            (x0c, y0c),
            x1c - x0c,
            y1c - y0c,
            fill=False,
            edgecolor=box_color,
            linewidth=lw,
        )
        ax.add_patch(rect)

        inset_pos = zspec.get("inset_pos", None)  # (x0,y0,w,h) in axes fraction
        if inset_pos is not None:
            axins = ax.inset_axes(inset_pos, transform=ax.transAxes)
        else:
            loc = zspec.get("loc", zoom_loc)
            frac = zspec.get("frac", zoom_inset_frac)
            axins = inset_axes(
                ax,
                width=f"{100*frac:.0f}%",
                height=f"{100*frac:.0f}%",
                loc=loc,
                borderpad=1.0,
            )

        axins.imshow(rgb, origin="lower")
        axins.set_xlim(x0c, x1c)
        axins.set_ylim(y0c, y1c)
        axins.set_xticks([])
        axins.set_yticks([])
        for spine in axins.spines.values():
            spine.set_edgecolor(box_color)
            spine.set_linewidth(lw)

        if mark:
            loc1 = zspec.get("mark_loc1", 2)
            loc2 = zspec.get("mark_loc2", 4)
            mark_inset(ax, axins, loc1=loc1, loc2=loc2, fc="none", ec=box_color, lw=lw)

    # legend
    legend_patches = [Patch(color=resolved_rgb[c], label=labels.get(c, c)) for c in valid_colors]
    leg = ax.legend(
        handles=legend_patches,
        loc="lower left",
        facecolor="black",
        framealpha=0.7,
        edgecolor="white",
        fontsize=8,
    )
    for t in leg.get_texts():
        t.set_color("white")

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    os.makedirs("PLOTS", exist_ok=True)
    pdf_path = f"PLOTS/{output_basename}_{attempt}.pdf"
    plt.savefig(pdf_path, bbox_inches="tight", dpi=600, facecolor="white")
    plt.close(fig)
    print(f"Saved composite PDF preview: {pdf_path}")


    return rgb, wcs_out


In [ ]:
color_to_lines = {
    "blue":  "OIII5007flux",
    "green": "Haflux",
    "red":   ["NII6584flux", "SII6716flux", "SII6731flux"],
}

color_labels = {
    "blue": r"[O III] $\lambda$5007",
    "green": r"H$\alpha$ $\lambda$6563",
    "red": r"[N II]$\lambda$6584 + [S II]$\lambda\lambda$6716,6731",    
}

channel_percentile = {
    "blue":  99.5,   
    "green": 99.5,  
    "red":   99.5,   
}

channel_gamma = {
    "blue":  0.6,    
    "green": 0.6,
    "red":   0.6,
}

zooms = [
    dict(center_pix=(3000, 3500), size=700, inset_pos=(0.05, 0.1, 0.3, 0.3), color="white"),
    dict(center_pix=(3200, 5500), size=700, inset_pos=(0.75, 0.65, 0.3, 0.3), color="white"),
]
 
channel_sky_percentile = {
    "blue":  2.0,
    "green": 5.0,
    "red":   10.0,
}

rgb, wcs_out = save_color_mosaic(
    directory_maps="../M33-Maps-Calibrated/",
    color_to_lines=color_to_lines,
    labels=color_labels,
    attempt="v40",
    zooms=zooms,
    output_basename="M33_Mosaic",
    channel_percentile=channel_percentile,
    channel_sky_percentile=channel_sky_percentile,
    channel_gamma=channel_gamma,
) 


Saved composite PDF preview: PLOTS/M33_Mosaic_v40.pdf
